# **Space X Falcon 9 First Stage Landing Prediction**

## Lab 2: Data Wrangling

Estimated time needed: **60** minutes

In this lab, we will perform Exploratory Data Analysis (EDA) to find patterns in the data
and determine the label for training supervised models.

The booster landing outcomes include:
- `True Ocean` — successfully landed in a specific ocean region
- `False Ocean` — unsuccessfully landed in a specific ocean region
- `True RTLS` — successfully landed on a ground pad
- `False RTLS` — unsuccessfully landed on a ground pad
- `True ASDS` — successfully landed on a drone ship
- `False ASDS` — unsuccessfully landed on a drone ship
- `None ASDS` / `None None` — failure to land

### Objectives

- Perform exploratory Data Analysis
- Determine Training Labels (`Class`: 1 = success, 0 = failure)

---
## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
print('Libraries loaded.')

Libraries loaded.


## Load the Dataset

Load the SpaceX dataset from the previous section.

In [2]:
# Load from IBM cloud URL for consistency (same data as our API collection output)
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv")
df.head(10)

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
5,6,2014-01-06,Falcon 9,3325.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1005,-80.577366,28.561857
6,7,2014-04-18,Falcon 9,2296.000000,ISS,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1006,-80.577366,28.561857
7,8,2014-07-14,Falcon 9,1316.000000,LEO,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1007,-80.577366,28.561857
8,9,2014-08-05,Falcon 9,4535.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1008,-80.577366,28.561857
9,10,2014-09-07,Falcon 9,4428.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1011,-80.577366,28.561857


## Identify Missing Values

In [3]:
# Calculate the percentage of missing values in each attribute
missing_pct = df.isnull().sum() / len(df) * 100
print('Missing value percentages:')
print(missing_pct)

Missing value percentages:
FlightNumber       0.000000
Date               0.000000
BoosterVersion     0.000000
PayloadMass        0.000000
Orbit              0.000000
LaunchSite         0.000000
Outcome            0.000000
Flights            0.000000
GridFins           0.000000
Reused             0.000000
Legs               0.000000
LandingPad        28.888889
Block              0.000000
ReusedCount        0.000000
Serial             0.000000
Longitude          0.000000
Latitude           0.000000
dtype: float64


In [4]:
# Identify numerical and categorical columns
print('Data types:')
print(df.dtypes)

Data types:
FlightNumber        int64
Date               object
BoosterVersion     object
PayloadMass       float64
Orbit              object
LaunchSite         object
Outcome            object
Flights             int64
GridFins             bool
Reused               bool
Legs                 bool
LandingPad         object
Block             float64
ReusedCount         int64
Serial             object
Longitude         float64
Latitude          float64
dtype: object


## TASK 1: Calculate the Number of Launches on Each Site

In [5]:
# Number of launches for each site
print(df['LaunchSite'].value_counts())

LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64


## TASK 2: Calculate the Number and Occurrence of Each Orbit

In [6]:
# Number and occurrence of each orbit
print(df['Orbit'].value_counts())

Orbit
GTO      27
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
ES-L1     1
HEO       1
SO        1
GEO       1
Name: count, dtype: int64


## TASK 3: Calculate the Number and Occurrence of Mission Outcome per Orbit

In [7]:
# Landing outcomes
landing_outcomes = df['Outcome'].value_counts()
print(landing_outcomes)

Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Name: count, dtype: int64


In [8]:
for i, outcome in enumerate(landing_outcomes.keys()):
    print(i, outcome)

0 True ASDS
1 None None
2 True RTLS
3 False ASDS
4 True Ocean
5 False Ocean
6 None ASDS
7 False RTLS


In [9]:
# Define the set of bad outcomes (unsuccessful landings)
bad_outcomes = set(landing_outcomes.keys()[[1, 3, 5, 6, 7]])
print('Bad outcomes:')
print(bad_outcomes)

Bad outcomes:
{'False ASDS', 'None ASDS', 'False RTLS', 'False Ocean', 'None None'}


## TASK 4: Create a Landing Outcome Label from Outcome Column

Create a `Class` column:
- `1` if the outcome is **not** in `bad_outcomes` (successful landing)
- `0` if the outcome **is** in `bad_outcomes` (unsuccessful landing)

In [10]:
# Create the landing class label
landing_class = [0 if outcome in bad_outcomes else 1 for outcome in df['Outcome']]

df['Class'] = landing_class
df[['Class']].head(8)

,Class
0,0
1,0
2,0
3,0
4,0
5,0
6,1
7,1


In [11]:
df.head(5)

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


In [12]:
# Success rate
success_rate = df['Class'].mean()
print(f'Overall landing success rate: {success_rate:.4f} ({success_rate:.2%})')

Overall landing success rate: 0.6667 (66.67%)


## Save to CSV

Export the wrangled dataset for downstream EDA and modeling.

In [13]:
os.makedirs('../data', exist_ok=True)
output_path = '../data/dataset_part_2.csv'
df.to_csv(output_path, index=False)
print(f'Saved {len(df)} rows to {output_path}')

Saved 90 rows to ../data/dataset_part_2.csv


## Summary

In this notebook we:

1. **Loaded** the API-collected dataset (`dataset_part_1.csv`)
2. **Identified** missing values and data types
3. **Counted** launches per site and per orbit
4. **Analyzed** landing outcomes and identified unsuccessful landing categories
5. **Created** the binary classification label `Class` (1 = successful landing, 0 = unsuccessful)
6. **Saved** the wrangled dataset as `data/dataset_part_2.csv`

The overall landing success rate is approximately 66.7%.